# 예제 03. 고객 세분화 및 예금 유치 대시보드
### 예제 02의 분석 결과를 한 화면에 시각화 · 1일 차 · 2시간

> **실행 방법**: 위에서부터 순서대로 실행하세요 (`Shift + Enter`).

| 항목 | 내용 |
|---|---|
| 데이터 | 예제 02에서 저장한 `bank_customers_prepared.csv` (UCI Bank Marketing 45,211건 + 직업·연령대·잔액구간) |
| 오늘의 결과물 | 직업군을 고르면 그래프가 바뀌는 **인터랙티브 고객 분석 대시보드** + 고객관리 개선 아이디어 |

**오늘 해결할 질문**: "고객군별 가입 패턴을 한 화면에서 비교하고, 고객관리 아이디어로 연결할 수 있을까?"

> ⚠️ 예제 02와 같은 약속입니다. 포르투갈 은행의 2008~2010년 기록이며, 집단 단위 통계로만 읽습니다.

## 0. 실습 준비

아래 셀을 실행하면 실습 데이터를 **GitHub에서 자동으로 내려받습니다.** (강의장 인터넷이 막혀 **[파일 선택]** 버튼이 나오면, 강사가 배포한 파일을 고르면 됩니다.)
`데이터 준비 완료: True`가 나오면 성공입니다.

In [ ]:
import os
import urllib.request
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILE = "bank_customers_prepared.csv"   # 예제 02에서 저장한 파일
if not os.path.exists(FILE):          # 파일이 없으면 GitHub에서 내려받습니다
    try:
        urllib.request.urlretrieve(DATA_URL + FILE, FILE)
    except Exception:                 # 인터넷이 막혔을 때만 직접 올리기
        from google.colab import files
        files.upload()
print("데이터 준비 완료:", os.path.exists(FILE))

## 03-1. 데이터 읽고 구간 순서 되살리기

CSV 파일은 글자만 저장하기 때문에 "20대 이하 → 30대 → …" 같은 **순서 정보가 사라집니다.**
그대로 그래프를 그리면 가나다순으로 섞이므로, 순서를 다시 알려 줍니다.

In [ ]:
df = pd.read_csv(FILE)

연령순서 = ["20대 이하", "30대", "40대", "50대", "60대 이상"]
잔액순서 = ["마이너스", "0유로", "1~1,000", "1,001~5,000", "5,000 초과"]
df["연령대"] = pd.Categorical(df["연령대"], categories=연령순서, ordered=True)
df["잔액구간"] = pd.Categorical(df["잔액구간"], categories=잔액순서, ordered=True)

print("고객 수:", len(df))
df[["직업", "연령대", "잔액구간", "y"]].head()

## 03-2. 대시보드용 항목 두 개 더 만들기

- **대출상태**: 주택담보대출(`housing`)과 개인대출(`loan`)을 합쳐 네 가지로 나눕니다.
- **과거캠페인**: 이전 캠페인 결과(`poutcome`)를 한국어로 바꿉니다.

`df.loc[조건, "항목"] = 값`은 "조건에 맞는 행의 항목만 값으로 바꾼다"는 뜻입니다.

In [ ]:
df["대출상태"] = "대출 없음"
df.loc[(df["housing"] == "yes") & (df["loan"] == "no"), "대출상태"] = "주택담보만"
df.loc[(df["housing"] == "no") & (df["loan"] == "yes"), "대출상태"] = "개인대출만"
df.loc[(df["housing"] == "yes") & (df["loan"] == "yes"), "대출상태"] = "둘 다"
df["대출상태"] = pd.Categorical(df["대출상태"], categories=["대출 없음", "주택담보만", "개인대출만", "둘 다"], ordered=True)

df["대출상태"].value_counts()

In [ ]:
캠페인명 = {"success": "이전 가입", "failure": "이전 미가입", "other": "기타", "unknown": "이전 연락 없음"}
df["과거캠페인"] = df["poutcome"].map(캠페인명)
df["과거캠페인"] = pd.Categorical(df["과거캠페인"], categories=list(캠페인명.values()), ordered=True)

df["과거캠페인"].value_counts()

## 03-3. 가입률표 함수 (예제 02와 같음)

예제 02의 `가입률표()`와 같은데, 이번에는 **어떤 데이터로 계산할지**도 함께 받습니다.
그래야 "학생 고객만", "은퇴자 고객만"처럼 일부 고객으로도 계산할 수 있습니다.

In [ ]:
def 가입률표(데이터, 기준):
    표 = 데이터.groupby(기준, observed=True)["y"].agg(
        고객수="size",
        가입자수=lambda x: (x == "yes").sum(),
    )
    표["가입률(%)"] = (표["가입자수"] / 표["고객수"] * 100).round(1)
    return 표

가입률표(df, "대출상태")

## 03-4. 핵심 지표 카드

대시보드 맨 위에는 보통 **한눈에 볼 숫자**를 둡니다: 고객 수 · 가입자 수 · 가입률.

In [ ]:
고객수 = len(df)
가입자수 = (df["y"] == "yes").sum()
전체가입률 = 가입자수 / 고객수 * 100

fig = go.Figure()
fig.add_trace(go.Indicator(mode="number", value=고객수, title="고객 수(명)", domain={"x": [0, 0.3]}))
fig.add_trace(go.Indicator(mode="number", value=가입자수, title="가입자 수(명)", domain={"x": [0.35, 0.65]}))
fig.add_trace(go.Indicator(mode="number", value=전체가입률, number={"suffix": "%", "valueformat": ".1f"},
                           title="정기예금 가입률", domain={"x": [0.7, 1]}))
fig.update_layout(height=200)
fig.show()

## 03-5. 고객군별 막대그래프 — 연령대

막대 위 숫자는 가입률, 마우스를 올리면 **고객수**도 보입니다. 점선은 전체 가입률(11.7%)입니다.

In [ ]:
표 = 가입률표(df, "연령대").reset_index()
fig = px.bar(표, x="연령대", y="가입률(%)", text="가입률(%)", hover_data=["고객수"],
             title="연령대별 정기예금 가입률")
fig.add_hline(y=전체가입률, line_dash="dot", annotation_text="전체 평균")
fig.show()

## 03-6. 잔액 구간 · 대출 상태별 막대그래프

같은 코드에서 `기준`만 바꾸면 됩니다.

In [ ]:
기준 = "잔액구간"      # ← "대출상태", "과거캠페인", "직업" 으로도 바꿔 보세요

표 = 가입률표(df, 기준).reset_index()
fig = px.bar(표, x=기준, y="가입률(%)", text="가입률(%)", hover_data=["고객수"],
             title=f"{기준}별 정기예금 가입률")
fig.add_hline(y=전체가입률, line_dash="dot", annotation_text="전체 평균")
fig.show()

> 💡 **핵심 정리** — 대출이 하나도 없는 고객의 가입률은 **18.2%**, 두 대출을 모두 가진 고객은 **6.1%**입니다.
> 잔액이 마이너스인 고객은 5.6%로 가장 낮습니다. "여유 자금이 있는지"가 가입과 함께 움직이는 것으로 보입니다.

## 03-7. 고객 세분화 — 연령대 × 잔액구간

두 기준을 **동시에** 나누면 더 작은 고객군이 생깁니다. 색이 진할수록 가입률이 높습니다.
아래 두 번째 표로 각 칸의 고객수도 꼭 확인하세요. 고객이 적은 칸은 가입률이 크게 흔들립니다.

In [ ]:
세분표 = 가입률표(df, ["연령대", "잔액구간"])
가입률판 = 세분표["가입률(%)"].unstack()      # unstack: 두 번째 기준을 가로(열)로 펼치기

fig = px.imshow(가입률판, text_auto=True, color_continuous_scale="Blues",
                title="연령대 × 잔액구간 정기예금 가입률(%)")
fig.show()

In [ ]:
세분표["고객수"].unstack()     # 각 칸의 고객 수

## 03-8. 한 화면 대시보드 만들기 ⭐

지금까지 만든 그래프 4개를 **한 화면(2×2)** 에 모읍니다.
`대시보드("직업")`처럼 직업을 넣으면 그 직업 고객만으로 다시 계산합니다.
함수 안의 코드는 읽기만 하고, 쓰는 방법(`대시보드("학생")`)만 기억하면 됩니다.

In [ ]:
def 대시보드(직업="전체"):
    데이터 = df if 직업 == "전체" else df[df["직업"] == 직업]
    기준들 = ["연령대", "잔액구간", "대출상태", "과거캠페인"]
    fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{k}별 가입률(%)" for k in 기준들])
    for i, 기준 in enumerate(기준들):
        표 = 가입률표(데이터, 기준).reset_index()
        fig.add_trace(go.Bar(x=표[기준], y=표["가입률(%)"], text=표["가입률(%)"], customdata=표["고객수"],
                             hovertemplate="%{x}<br>가입률 %{y}%<br>고객수 %{customdata}명<extra></extra>"),
                      row=i // 2 + 1, col=i % 2 + 1)
    가입률 = (데이터["y"] == "yes").mean() * 100
    fig.add_hline(y=가입률, line_dash="dot", row="all", col="all")
    fig.update_yaxes(range=[0, 100])
    fig.update_layout(title=f"[{직업}] 고객 {len(데이터):,}명 · 정기예금 가입률 {가입률:.1f}%",
                      showlegend=False, height=650)
    return fig

대시보드().show()

## 03-9. 🔧 직업군을 골라 보는 인터랙티브 대시보드

아래 셀을 실행하면 **선택 상자**가 나타납니다. 직업을 고르면 대시보드가 그 고객군으로 바뀝니다.
점선(그 고객군의 평균 가입률)과 제목의 고객 수를 함께 보세요.

In [ ]:
from ipywidgets import interact

직업목록 = ["전체"] + sorted(df["직업"].unique().tolist())

def 보기(직업="전체"):
    대시보드(직업).show()

interact(보기, 직업=직업목록);

## 03-10. 두 고객군 비교하기

선택 상자 대신 이름을 직접 넣어 두 고객군을 위아래로 비교합니다. 다른 직업으로 바꿔 보세요.

In [ ]:
대시보드("학생").show()
대시보드("생산·현장직").show()

**관찰 기록표** (더블클릭해서 직접 채워 보세요)

| 고객군 | 눈에 띄는 점 | 고객 수가 충분한가? |
|---|---|---|
| 학생 | | |
| 생산·현장직 | | |
| (내가 고른 고객군) | | |

## 03-11. [선택] 대시보드를 HTML 파일로 저장

저장한 HTML 파일은 인터넷이 되는 PC의 웹브라우저에서 열 수 있고, 마우스를 올리면 값도 보입니다.

In [ ]:
대시보드("전체").write_html("dashboard_전체.html", include_plotlyjs="cdn")

from google.colab import files
files.download("dashboard_전체.html")

## 03-12. 고객관리 개선 아이디어 만들기 💬

대시보드에서 찾은 사실을 **관찰 → 가설 → 아이디어 → 확인 방법** 순서로 정리합니다.
첫 줄은 예시입니다. 나머지는 조별로 채워 보세요.

| 관찰 (데이터로 확인한 사실) | 가설 (왜 그럴까?) | 개선 아이디어 | 확인 방법 |
|---|---|---|---|
| 이전 캠페인에 가입했던 고객의 가입률 64.7% (전체 11.7%) | 이미 거래해 본 고객은 상품을 신뢰한다 | 만기 도래 고객에게 재예치 안내를 먼저 한다 | 안내한 고객과 안 한 고객의 재예치율 비교 |
| | | | |
| | | | |

> ⚠️ **아이디어를 낼 때 지킬 것**
> - 함께 나타난 것은 원인이 아닙니다. 아이디어는 반드시 **확인 방법**과 짝을 이룹니다.
> - 고객 수가 적은 칸의 높은 가입률만 보고 결론 내리지 않습니다.
> - 연령·직업은 **상품과 안내 방식을 개선**하는 데 쓰고, 특정 개인을 자동으로 고르거나 배제하는 기준으로 쓰지 않습니다.

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 구간 순서 되살리기 | `pd.Categorical(값, categories=[...], ordered=True)` |
| 조건에 맞는 행만 바꾸기 | `df.loc[조건, "항목"] = 값` |
| 숫자 카드 | `go.Indicator(mode="number", value=...)` |
| 두 기준 표를 판으로 펼치기 | `표.unstack()` |
| 색으로 보는 표(히트맵) | `px.imshow(판, text_auto=True)` |
| 여러 그래프를 한 화면에 | `make_subplots(rows=2, cols=2)` |
| 선택 상자 | `interact(함수, 직업=목록)` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** 은퇴자 고객만으로 대시보드를 그려 보세요.
**이렇게 나오면 성공!** 제목이 `[은퇴자] 고객 2,264명 · 정기예금 가입률 22.8%`

**미션 2.** 연락 방법(`contact`)별 가입률표를 만들어 보세요.
**이렇게 나오면 성공!** cellular 14.9% · telephone 13.4% · unknown 4.1%

**미션 3.** 03-7의 세분화 판에서 가입률이 가장 높은 칸을 찾아보세요.
**이렇게 나오면 성공!** 60대 이상 × 1,001~5,000 → 37.1% (고객 706명)

**다음 예제 전에 이것만은 꼭!**
- [ ] `대시보드("직업")`으로 고객군을 바꿔 볼 수 있다
- [ ] 관찰 하나를 "관찰 → 가설 → 아이디어 → 확인 방법"으로 정리해 보았다

In [ ]:
# 미션 1
대시보드("학생").show()      # ← "학생"을 "은퇴자"로

In [ ]:
# 미션 2
가입률표(df, "직업")         # ← "직업"을 "contact"로

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답
대시보드("은퇴자").show()

In [ ]:
# 미션 2 정답
가입률표(df, "contact")

In [ ]:
# 미션 3 정답
세분표.sort_values("가입률(%)", ascending=False).head(3)

---
### ▶ 다음 예제 04. 금융 시계열 데이터 분석과 추세 시각화
고객 데이터에서 잠시 벗어나 **시간의 흐름**을 봅니다. 한국은행의 10년치 월별 예금금리로 상승·하락 구간과 추세를 찾아봅니다.